# RAG con NVIDIA NIM

Este notebook implementa un sistema **Retrieval-Augmented Generation (RAG)** completo, sin frameworks de alto nivel que oculten los internals.

**Stack:**
- **Embeddings:** NVIDIA NIM — `nvidia/nemotron-3-embed-1b`
- **Vector store (internals):** FAISS
- **Vector store (pipeline):** Qdrant en memoria
- **LLM de generación:** NVIDIA NIM — `nvidia/nemotron-3-super-120b-a12b`
- **Evaluación:** DeepEval usando el mismo endpoint NVIDIA NIM como juez

> El endpoint alojado de NVIDIA es compatible con el SDK de OpenAI. Solo cambia la URL base, la API key y el nombre del modelo.


| Parte | Tema |
|---|---|
| 0 | Instalación, API key y clientes NVIDIA NIM |
| 1 | El problema: LLM sin contexto |
| 2 | Internals: embeddings y búsqueda vectorial |
| 3 | Estrategias de chunking |
| 4 | RAG completo con Qdrant |
| 5 | Tuning de prompts y thresholds |
| 6 | Evaluación con DeepEval y NVIDIA NIM |


## Parte 0 — Preparación en Google Colab

No necesitas instalar un servidor local ni descargar pesos. La generación y los embeddings se consumen mediante los endpoints alojados de NVIDIA NIM.

1. Crea una clave en [NVIDIA Build](https://build.nvidia.com/).
2. En Colab abre **🔑 Secrets**, crea `NVIDIA_API_KEY` y habilita el acceso al notebook.
3. Ejecuta las celdas en orden.


In [1]:
# Dependencias del laboratorio
%pip install -qU openai qdrant-client faiss-cpu deepeval python-dotenv pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.2/880.2 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46

### 0.1 Configuración segura de la API key

La celda busca la clave primero en los Secrets de Colab, luego en la variable de entorno `NVIDIA_API_KEY` y, si no existe, la solicita sin mostrarla en pantalla.


In [2]:
import os
import json
import getpass
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()

if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

nim_client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)
print("✅ Cliente NVIDIA NIM configurado")


✅ Cliente NVIDIA NIM configurado


### 0.2 Embeddings con NVIDIA NIM

El modelo de embeddings convierte consultas y documentos en vectores. Se normalizan para que el producto interno de FAISS equivalga a similitud coseno.


In [3]:
class EmbeddingService:
    """Cliente pequeño para el endpoint de embeddings de NVIDIA NIM."""

    def __init__(self, client: OpenAI, model: str = EMBEDDING_MODEL):
        self.client = client
        self.model = model
        probe = self.embed(["prueba de dimensión"], input_type="passage")
        self.dim = len(probe[0])
        print(f"✅ Embeddings: {self.model} — dimensión={self.dim}")

    def embed(self, texts: List[str], input_type: str = "passage") -> List[List[float]]:
        if not texts:
            return []
        response = self.client.embeddings.create(
            model=self.model,
            input=texts,
            encoding_format="float",
            extra_body={"input_type": input_type, "truncate": "END"},
        )
        vectors = np.asarray([item.embedding for item in response.data], dtype="float32")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        vectors = vectors / np.clip(norms, 1e-12, None)
        return vectors.tolist()


embedding_service = EmbeddingService(nim_client)


✅ Embeddings: nvidia/nemotron-3-embed-1b — dimensión=2048


### 0.3 Generación con Nemotron 3 Super

La clase encapsula el endpoint de chat para que el resto del notebook no dependa del proveedor. Se desactiva el bloque de razonamiento visible y se devuelve únicamente la respuesta final.


In [4]:
class GenerationService:
    """Generación de texto mediante el endpoint OpenAI-compatible de NVIDIA NIM."""

    def __init__(self, client: OpenAI, model: str = GENERATION_MODEL):
        self.client = client
        self.model = model
        print(f"✅ Generación: {self.model}")

    def generate(self, prompt: str, temperature: float = 0.2, max_tokens: int = 700) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            top_p=0.95,
            max_tokens=max_tokens,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        return (response.choices[0].message.content or "").strip()


generation_service = GenerationService(nim_client)
print("Prueba:", generation_service.generate("Responde únicamente: hola", max_tokens=20))


✅ Generación: nvidia/nemotron-3-super-120b-a12b
Prueba: hola


### 0.4 Comprobación de la configuración

Los dos modelos comparten la misma API key y URL base, pero cumplen tareas distintas: el modelo *embed* recupera información y Nemotron 3 Super genera o evalúa texto.


In [5]:
print("Configuración activa")
print("- Endpoint:  ", NVIDIA_BASE_URL)
print("- Generación:", GENERATION_MODEL)
print("- Embeddings: ", EMBEDDING_MODEL)


Configuración activa
- Endpoint:   https://integrate.api.nvidia.com/v1
- Generación: nvidia/nemotron-3-super-120b-a12b
- Embeddings:  nvidia/nemotron-3-embed-1b


> Las llamadas remotas consumen cuota del endpoint. FAISS y Qdrant se ejecutan dentro del runtime de Colab; no requieren una cuenta ni un servidor adicional.


> **Arquitectura:** la consulta se convierte en embedding con Nemotron Embed, Qdrant recupera los fragmentos más similares y Nemotron 3 Super redacta la respuesta usando únicamente ese contexto.


### 0.5 Datos del curso

Usaremos documentos internos ficticios. Si subes `data/company_documents.json`, el notebook los carga; si no, crea automáticamente un conjunto mínimo para que el laboratorio funcione en Colab.


In [6]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: str | Path) -> List[Dict]:
    path = Path(path)
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS
    with path.open(encoding="utf-8") as file:
        documents = json.load(file)
    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


In [8]:
DOCS_PATH = Path("company_documents.json")
company_documents = load_documents(DOCS_PATH)

print(f"✅ {len(company_documents)} documentos disponibles")
for doc in company_documents:
    metadata = doc.get("metadata", {})
    print(f"  [{doc['id']}] {doc['title']} — {metadata.get('department', '—')}")


✅ 4 documentos disponibles
  [product_001] CloudSync Pro — Plan Enterprise — product
  [policy_001] Política de Trabajo Remoto 2024 — hr
  [process_001] Proceso de Reembolso a Clientes — support
  [guide_001] Checklist de Onboarding para Nuevos Empleados — hr


In [9]:
# ── Filtrado por metadatos (opcional) ────────────────────────

# Ejemplo: cargar solo documentos de RRHH
hr_docs = [d for d in company_documents if d.get("metadata", {}).get("department") == "hr"]
print(f"\nDocumentos de RRHH: {len(hr_docs)}")
for d in hr_docs:
    print(f"  - {d['title']}")


Documentos de RRHH: 2
  - Política de Trabajo Remoto 2024
  - Checklist de Onboarding para Nuevos Empleados


## Parte 1 — El problema: LLM sin contexto
Antes de construir RAG, veamos qué pasa cuando le hacemos preguntas al modelo
sobre información que no está en su entrenamiento.

In [10]:
test_questions = [
    "¿Cuál es el precio del plan Enterprise de CloudSync Pro?",
    "¿Cuántos días a la semana pueden trabajar remotamente los empleados?",
    "¿Qué pasa con los reembolsos mayores a $100?",
    "¿Qué ocurre durante la primera semana de onboarding?",
]

In [11]:

print("PREGUNTAS AL MODELO SIN CONTEXTO")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\nPregunta {i}: {question}")
    response = generation_service.generate(
        f"Responde brevemente: {question}",
        temperature=0.0,
    )
    print(f"Respuesta: {response.strip()}")
    print("-" * 40)

PREGUNTAS AL MODELO SIN CONTEXTO

Pregunta 1: ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
Respuesta: Lo siento, pero no dispongo de información específica sobre el precio del plan Enterprise de CloudSync Pro, ya que no es un producto ampliamente documentado en fuentes públicas o en mi base de datos de conocimiento actual (hasta 2024). Los precios de planes empresariales suelen variar según región, número de usuarios, características personalizadas y acuerdos de volumen, y a menudo se proporcionan bajo solicitud directa al proveedor.

Te recomiendo visitar el sitio web oficial de CloudSync Pro o contactar directamente a su equipo de ventas para obtener un presupuesto actualizado y adaptado a tus necesidades.
----------------------------------------

Pregunta 2: ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
Respuesta: La cantidad de días a la semana que los empleados pueden trabajar remotamente depende de las políticas específicas de cada empresa, indu

**¿Qué observas?**

- El modelo responde con información genérica, inventada o simplemente dice que no sabe.

- Esto es exactamente el problema que RAG resuelve: darle al modelo acceso a **tu información específica** en el momento de la consulta.

## Parte 2 — Internals: embeddings y vector store con FAISS
Antes de usar Qdrant, vamos a construir un vector store desde cero con FAISS.
El objetivo es entender exactamente qué pasa por dentro.
### 2.1 ¿Qué es un embedding?
Un embedding es una función que convierte texto en un vector de números reales.
Textos semánticamente similares quedan cerca en ese espacio vectorial.

In [12]:
# Veamos embeddings en acción
sample_texts = [
    "El gato duerme en el sofá",
    "El felino descansa en el mueble",   # semánticamente similar al anterior
    "La política de reembolsos es clara",  # distinto tema
]

vectors = embedding_service.embed(sample_texts, input_type="passage")

print(f"Dimensión de cada vector: {len(vectors[0])}")
print(f"Primeros 5 valores del vector 1: {[round(v, 4) for v in vectors[0][:5]]}")


# Calcular similitud coseno manualmente entre los tres textos
def cosine_sim(a: List[float], b: List[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sim_01 = cosine_sim(vectors[0], vectors[1])
sim_02 = cosine_sim(vectors[0], vectors[2])

print(f"\nSimilitud coseno:")
print(f"  texto 0 ↔ texto 1 (similar):  {sim_01:.4f}")
print(f"  texto 0 ↔ texto 2 (distinto): {sim_02:.4f}")

Dimensión de cada vector: 2048
Primeros 5 valores del vector 1: [0.1533, 0.004, -0.0594, -0.0096, -0.0398]

Similitud coseno:
  texto 0 ↔ texto 1 (similar):  0.7850
  texto 0 ↔ texto 2 (distinto): 0.4590


### 2.2 Vector store manual con FAISS

FAISS es una librería de Facebook para búsqueda eficiente de vectores similares.
Lo usamos aquí para entender los internals; en producción usaremos Qdrant.

In [13]:
!pip install faiss-cpu


In [14]:
import faiss

@dataclass
class DocumentChunk:
    """Representa un fragmento de documento con su vector y metadatos."""
    id: str
    content: str
    source_doc_id: str
    title: str
    chunk_index: int
    department: str = ""
    doc_type: str = ""
    embedding: Optional[List[float]] = None


class FAISSVectorStore:
    """
    Vector store minimalista construido sobre FAISS.
    Propósito: entender los internals de búsqueda vectorial.
    Para producción → ver Parte 4 con Qdrant.
    """

    def __init__(self, dim: int):
        # IndexFlatIP = producto interno (equivale a coseno si los vectores están normalizados)
        self.index = faiss.IndexFlatIP(dim)
        self.documents: List[DocumentChunk] = []
        self.id_to_index: Dict[str, int] = {}
        self.dim = dim

    def add(self, docs: List[DocumentChunk]) -> None:
        embeddings = []
        for doc in docs:
            if doc.embedding is None:
                raise ValueError(f"Documento {doc.id} no tiene embedding")

            # Normalizar → la búsqueda por producto interno equivale a coseno
            vec = np.array(doc.embedding, dtype="float32")
            vec /= np.linalg.norm(vec)

            pos = len(self.documents)
            self.documents.append(doc)
            self.id_to_index[doc.id] = pos
            embeddings.append(vec)

        self.index.add(np.stack(embeddings))
        print(f"  ➕ {len(docs)} chunks añadidos (total: {self.index.ntotal})")

    def search(
        self,
        query_embedding: List[float],
        k: int = 3,
        threshold: float = 0.3,
    ) -> List[Tuple[DocumentChunk, float]]:
        if self.index.ntotal == 0:
            return []

        q = np.array(query_embedding, dtype="float32")
        q /= np.linalg.norm(q)

        scores, indices = self.index.search(q.reshape(1, -1), k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and score >= threshold:
                results.append((self.documents[idx], float(score)))
        return results

In [15]:
# Prueba rápida
faiss_store = FAISSVectorStore(dim=embedding_service.dim)

In [16]:
# Embeddear y añadir el primer documento como ejemplo
test_chunks_text = [
    "CloudSync Pro Enterprise cuesta $49/mes por usuario con compromiso anual.",
    "El plan mensual sin compromiso está disponible a $59/mes por usuario.",
]
test_vecs = embedding_service.embed(test_chunks_text, input_type="passage")


In [17]:
test_chunks = [
    DocumentChunk(
        id=f"test_{i}",
        content=text,
        source_doc_id="product_001",
        title="CloudSync Pro",
        chunk_index=i,
        embedding=vec,
    )
    for i, (text, vec) in enumerate(zip(test_chunks_text, test_vecs))
]
faiss_store.add(test_chunks)

  ➕ 2 chunks añadidos (total: 2)


In [18]:
# Búsqueda de prueba
query_vec = embedding_service.embed(["¿Cuánto cuesta el plan anual?"], input_type="query")[0]
results = faiss_store.search(query_vec, k=2, threshold=0.2)

print("\nResultados de búsqueda en FAISS:")
for doc, score in results:
    print(f"  [{score:.4f}] {doc.content}")


Resultados de búsqueda en FAISS:
  [0.3349] El plan mensual sin compromiso está disponible a $59/mes por usuario.
  [0.3229] CloudSync Pro Enterprise cuesta $49/mes por usuario con compromiso anual.


## Parte 3 — Estrategias de chunking

No podemos meter documentos completos en cada query: hay límites de contexto y el ruido baja la calidad de las respuestas. Necesitamos dividir los documentos en fragmentos (*chunks*) antes de indexarlos.

Vamos a comparar dos estrategias:


In [19]:
def simple_chunker(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Chunking por caracteres con overlap.
    Ventaja: tamaño predecible.
    Desventaja: puede cortar a mitad de oración.
    """
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Intentar terminar en punto si es posible
        if end < len(text):
            last_period = text.rfind(".", start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        next_start = end - overlap
        start = max(next_start, start + 1)
        if end >= len(text):
            break
    return chunks

In [20]:
def semantic_chunker(text: str, max_chunk_size: int = 400) -> List[str]:
    """
    Chunking semántico: respeta párrafos y oraciones.
    Ventaja: preserva unidades de significado.
    Desventaja: tamaño variable.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    # Si no hay doble salto de línea, dividir por oraciones
    if len(paragraphs) == 1:
        paragraphs = [s.strip() + "." for s in text.split(".") if s.strip()]

    chunks, current = [], ""
    for para in paragraphs:
        if current and len(current) + len(para) + 2 > max_chunk_size:
            chunks.append(current.strip())
            current = para
        else:
            current = (current + "\n\n" + para).strip() if current else para

    if current.strip():
        chunks.append(current.strip())
    return chunks

In [21]:
# ── Comparación sobre el documento de pricing ──
sample = company_documents[0]
print(f"Documento: '{sample['title']}'")
print(f"Longitud original: {len(sample['content'])} caracteres\n")

simple_chunks = simple_chunker(sample["content"], chunk_size=200, overlap=40)
print(f"1. Chunking simple → {len(simple_chunks)} chunks:")
for i, c in enumerate(simple_chunks):
    print(f"   Chunk {i+1} ({len(c)} chars): {c[:80]}...")

semantic_chunks = semantic_chunker(sample["content"], max_chunk_size=250)
print(f"\n2. Chunking semántico → {len(semantic_chunks)} chunks:")
for i, c in enumerate(semantic_chunks):
    print(f"   Chunk {i+1} ({len(c)} chars): {c[:80]}...")

print("\n📌 Trade-offs:")
print("  Simple:   tamaño predecible, puede romper oraciones")
print("  Semántico: preserva significado, tamaño variable")

Documento: 'CloudSync Pro — Plan Enterprise'
Longitud original: 372 caracteres

1. Chunking simple → 3 chunks:
   Chunk 1 (187 chars): CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo...
   Chunk 2 (155 chars): 49/mes por usuario con compromiso anual. Incluye: control de versiones avanzado,...
   Chunk 3 (110 chars): n SSO, y SLA de 99.9% de disponibilidad. El plan mensual sin compromiso está dis...

2. Chunking semántico → 2 chunks:
   Chunk 1 (188 chars): CloudSync Pro Enterprise ofrece almacenamiento ilimitado, colaboración en tiempo...
   Chunk 2 (187 chars): Incluye: control de versiones avanzado, registros de auditoría, integración SSO,...

📌 Trade-offs:
  Simple:   tamaño predecible, puede romper oraciones
  Semántico: preserva significado, tamaño variable


## Parte 4 — RAG completo con Qdrant

Ahora construimos el pipeline RAG de producción usando Qdrant como vector store.
Qdrant ofrece filtrado por metadatos, persistencia y una API limpia.

Usamos el modo **in-memory** para el lab (sin servidor externo).

In [22]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION_NAME = "company_docs"

In [23]:
# Cliente in-memory — en producción: QdrantClient(url="http://localhost:6333")
qdrant = QdrantClient(":memory:")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_service.dim,
        distance=Distance.COSINE,
    ),
)
print(f"Colección '{COLLECTION_NAME}' creada (dim={embedding_service.dim})")

Colección 'company_docs' creada (dim=2048)


In [24]:
def ingest_documents(
    documents: List[Dict],
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    collection: str = COLLECTION_NAME,
    max_chunk_size: int = 350,
) -> int:
    """
    Pipeline de ingestión:
      1. Chunkear cada documento con chunking semántico
      2. Embeddear todos los chunks en batch
      3. Insertar en Qdrant con metadatos

    Retorna el número total de chunks insertados.
    """
    all_texts, all_meta = [], []

    for doc in documents:
        chunks = semantic_chunker(doc["content"], max_chunk_size=max_chunk_size)
        for i, chunk_text in enumerate(chunks):
            if len(chunk_text) < 20:   # descartar chunks vacíos
                continue
            all_texts.append(chunk_text)
            all_meta.append({
                "chunk_id": f"{doc['id']}_chunk_{i}",
                "source_doc_id": doc["id"],
                "title": doc["title"],
                "chunk_index": i,
                "department": doc["metadata"].get("department", ""),
                "doc_type": doc["metadata"].get("type", ""),
                "content": chunk_text,   # guardamos el texto para recuperarlo
            })

    # Embeddear en batch (más eficiente que uno por uno)
    print(f"  Generando embeddings para {len(all_texts)} chunks...")
    vectors = emb_service.embed(all_texts, input_type="passage")

    # Insertar en Qdrant
    points = [
        PointStruct(id=idx, vector=vec, payload=meta)
        for idx, (vec, meta) in enumerate(zip(vectors, all_meta))
    ]
    qdrant_client.upsert(collection_name=collection, points=points)

    print(f"  ✅ {len(points)} chunks indexados en Qdrant")
    return len(points)

In [25]:
total = ingest_documents(company_documents, qdrant, embedding_service)
print(f"\nTotal chunks en el índice: {total}")

  Generando embeddings para 7 chunks...
  ✅ 7 chunks indexados en Qdrant

Total chunks en el índice: 7


In [26]:
def retrieve(
    question: str,
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    k: int = 3,
    score_threshold: float = 0.3,
    collection: str = COLLECTION_NAME,
) -> List[Dict]:
    """
    Recupera los k chunks más relevantes para la pregunta.
    Retorna lista de payloads (con 'content', 'title', 'score').
    """
    query_vec = emb_service.embed([question], input_type="query")[0]

    hits = qdrant_client.query_points(
        collection_name=collection,
        query=query_vec,
        limit=k,
        score_threshold=score_threshold,
    ).points

    results = []
    for hit in hits:
        payload = hit.payload.copy()
        payload["score"] = hit.score
        results.append(payload)
    return results

In [27]:
def ask(
    question: str,
    qdrant_client: QdrantClient,
    emb_service: EmbeddingService,
    temperature: float = 0.1,
    k: int = 3,
    score_threshold: float = 0.3,
    system_role: str = "asistente empresarial",
) -> Tuple[str, List[Dict]]:
    """
    Pipeline RAG completo:
      1. Retrieve — buscar chunks relevantes
      2. Augment  — construir prompt con contexto
      3. Generate — llamar a GenerationService

    Retorna (respuesta, chunks_usados) para poder evaluar.
    """
    # 1. RETRIEVE
    chunks = retrieve(question, qdrant_client, emb_service, k=k, score_threshold=score_threshold)
    if not chunks:
        return "No encontré información relevante en los documentos.", []

    # 2. AUGMENT
    context = "\n\n---\n\n".join(
        f"[{c['title']}]\n{c['content']}" for c in chunks
    )
    prompt = f"""Eres un {system_role}. Responde la pregunta basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:"""

    # 3. GENERATE
    answer = generation_service.generate(prompt, temperature=temperature)
    return answer.strip(), chunks

In [28]:
# ── Prueba del pipeline completo ──
print("=" * 60)
print("RAG CON QDRANT — PIPELINE COMPLETO")
print("=" * 60)

for question in test_questions:
    answer, used_chunks = ask(question, qdrant, embedding_service)
    sources = list({c["title"] for c in used_chunks})
    print(f"\n❓ {question}")
    print(f"💬 {answer}")
    print(f"📎 Fuentes: {sources}")
    print("-" * 40)

RAG CON QDRANT — PIPELINE COMPLETO

❓ ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
💬 El precio del plan Enterprise de CloudSync Pro es de $49/mes por usuario con compromiso anual.
📎 Fuentes: ['CloudSync Pro — Plan Enterprise']
----------------------------------------

❓ ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
💬 Los empleados pueden trabajar de forma remota hasta 3 días por semana.
📎 Fuentes: ['Política de Trabajo Remoto 2024']
----------------------------------------

❓ ¿Qué pasa con los reembolsos mayores a $100?
💬 Los reembolsos mayores a $100 requieren aprobación del supervisor.
📎 Fuentes: ['Proceso de Reembolso a Clientes']
----------------------------------------

❓ ¿Qué ocurre durante la primera semana de onboarding?
💬 Durante la primera semana de onboarding, se completan los módulos de entrenamiento obligatorios (seguridad, cumplimiento, cultura).
📎 Fuentes: ['Checklist de Onboarding para Nuevos Empleados']
-------------------------------

## Parte 5 — Tuning: prompts y similarity threshold

Dos palancas principales para mejorar la calidad del RAG:
1. **El prompt** — el rol y las instrucciones que le damos al modelo
2. **El threshold** — cuán estrictos somos al filtrar chunks por relevancia

In [29]:
# ── 5.1 Efecto del rol en el prompt ──
question = "¿Cuál es la política de trabajo remoto?"

print("MISMO CONTEXTO, DIFERENTE ROL EN EL PROMPT\n")

roles = {
    "asistente de RRHH formal": 0.1,
    "agente de atención al cliente amigable": 0.4,
}

for role, temp in roles.items():
    answer, _ = ask(question, qdrant, embedding_service, temperature=temp, system_role=role)
    print(f"🎭 Rol: {role} (temp={temp})")
    print(f"   {answer}\n")

MISMO CONTEXTO, DIFERENTE ROL EN EL PROMPT

🎭 Rol: asistente de RRHH formal (temp=0.1)
   La política de trabajo remoto permite a los empleados trabajar de forma remota hasta 3 días por semana, requiere aprobación del jefe directo, incluye un estipendio de equipos de $500 anuales para adecuación de oficina en casa y establece que las videollamadas son obligatorias en reuniones de equipo.

🎭 Rol: agente de atención al cliente amigable (temp=0.4)
   La política de trabajo remoto permite a los empleados trabajar de forma remota hasta 3 días por semana, requiere aprobación del jefe directo, incluye un estipendio de equipos de $500 anuales para adecuación de oficina en casa y exige que las videollamadas sean obligatorias en reuniones de equipo. Esta política está vigente desde enero de 2024.



In [30]:
# ── 5.2 Efecto del similarity threshold ──
query_test = "trabajo remoto empleados"
query_vec = embedding_service.embed([query_test], input_type="query")[0]

print(f"EFECTO DEL THRESHOLD — query: '{query_test}'\n")
print(f"{'Threshold':>10} | {'Resultados':>10} | Scores")
print("-" * 50)

for threshold in [0.1, 0.3, 0.5, 0.7]:
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=5,
        score_threshold=threshold,
    ).points
    scores = [round(h.score, 3) for h in hits]
    print(f"{threshold:>10} | {len(hits):>10} | {scores}")

print("""
Regla práctica:
  < 0.3  → demasiado ruido, contexto irrelevante
  0.3–0.5 → buen balance para documentos empresariales
  > 0.5  → puede perder resultados relevantes con distinto vocabulario
""")

EFECTO DEL THRESHOLD — query: 'trabajo remoto empleados'

 Threshold | Resultados | Scores
--------------------------------------------------
       0.1 |          5 | [0.488, 0.237, 0.176, 0.11, 0.104]
       0.3 |          1 | [0.488]
       0.5 |          0 | []
       0.7 |          0 | []

Regla práctica:
  < 0.3  → demasiado ruido, contexto irrelevante
  0.3–0.5 → buen balance para documentos empresariales
  > 0.5  → puede perder resultados relevantes con distinto vocabulario



## Parte 6 — Evaluación con DeepEval y NVIDIA NIM

DeepEval usa un LLM como juez. El adaptador siguiente implementa su interfaz `DeepEvalBaseLLM` y llama al mismo endpoint NVIDIA NIM. Cuando DeepEval solicita una estructura Pydantic, el adaptador fuerza JSON y valida la respuesta.

> Esta sección genera varias llamadas y puede consumir la cuota gratuita. Para una demostración rápida, evalúa primero uno o dos casos.


In [31]:
from typing import Type
from pydantic import BaseModel
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate


class NvidiaNIMEvalModel(DeepEvalBaseLLM):
    def __init__(self, client: OpenAI, model: str = GENERATION_MODEL):
        self.client = client
        self.model = model

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema: Optional[Type[BaseModel]] = None):
        messages = [{"role": "user", "content": prompt}]
        kwargs = {}
        if schema is not None:
            schema_json = json.dumps(schema.model_json_schema(), ensure_ascii=False)
            messages[0]["content"] += (
                "\n\nDevuelve exclusivamente JSON válido que cumpla este esquema:\n" + schema_json
            )
            kwargs["response_format"] = {"type": "json_object"}

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.0,
            max_tokens=1200,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            **kwargs,
        )
        text = (response.choices[0].message.content or "").strip()
        return schema.model_validate_json(text) if schema is not None else text

    async def a_generate(self, prompt: str, schema: Optional[Type[BaseModel]] = None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return f"NVIDIA NIM — {self.model}"


/tmp/ipykernel_1789/3590622371.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [32]:
eval_model = NvidiaNIMEvalModel(nim_client)
print("✅ Modelo juez:", eval_model.get_model_name())


✅ Modelo juez: NVIDIA NIM — nvidia/nemotron-3-super-120b-a12b


> Generación, embeddings y evaluación quedan centralizados en NVIDIA NIM, sin procesos locales adicionales.


### 6.1 Evaluación sin *ground truth*

`FaithfulnessMetric` y `AnswerRelevancyMetric` necesitan la pregunta, la respuesta y los fragmentos recuperados, pero no una respuesta esperada.


In [33]:
# async_mode=False evita ráfagas de solicitudes contra el endpoint compartido.
faithfulness_metric = FaithfulnessMetric(
    model=eval_model, threshold=0.7, async_mode=False
)
answer_relevancy_metric = AnswerRelevancyMetric(
    model=eval_model, threshold=0.7, async_mode=False
)


In [34]:
# ── Construir test cases ──────────────────────────────────────
print("Generando respuestas RAG...")
test_cases_no_gt = []
for question in test_questions:
    answer, chunks = ask(question, qdrant, embedding_service)
    test_cases_no_gt.append(LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[c["content"] for c in chunks],
        # sin expected_output
    ))
    print(f"  ✅ '{question[:55]}...'")

Generando respuestas RAG...
  ✅ '¿Cuál es el precio del plan Enterprise de CloudSync Pro...'
  ✅ '¿Cuántos días a la semana pueden trabajar remotamente l...'
  ✅ '¿Qué pasa con los reembolsos mayores a $100?...'
  ✅ '¿Qué ocurre durante la primera semana de onboarding?...'


In [35]:
print("\nEjecutando evaluación...")
results_no_gt = evaluate(
    test_cases=test_cases_no_gt,
    metrics=[faithfulness_metric, answer_relevancy_metric],
)


Ejecutando evaluación...


✨ You're running DeepEval's latest Faithfulness Metric! (using NVIDIA NIM — nvidia/nemotron-3-super-120b-a12b, 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using NVIDIA NIM — nvidia/nemotron-3-super-120b-a12b,
strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Faithfulness               │ 1.00                  │ 100.00% | passed=4 | failed=0                 │ 4         │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=4 | failed=0                 │ 4         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=542318;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 51.86s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 100.0% | Passed: 4 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [36]:
ground_truths = [
    # ¿Cuál es el precio del plan Enterprise de CloudSync Pro?
    "El plan Enterprise de CloudSync Pro cuesta $49/mes por usuario con compromiso anual. "
    "Sin compromiso, el plan mensual está disponible a $59/mes por usuario.",

    # ¿Cuántos días a la semana pueden trabajar remotamente los empleados?
    "Los empleados pueden trabajar de forma remota hasta 3 días por semana, "
    "con aprobación del jefe directo.",

    # ¿Qué pasa con los reembolsos mayores a $100?
    "Los reembolsos mayores a $100 requieren aprobación del supervisor antes de procesarse. "
    "Una vez aprobados, se procesan en 3–5 días hábiles al método de pago original.",

    # ¿Qué ocurre durante la primera semana de onboarding?
    "Durante la primera semana se completan los módulos de entrenamiento obligatorios "
    "de seguridad, cumplimiento y cultura de la empresa.",
]

In [37]:
faithfulness_metric     = FaithfulnessMetric(model=eval_model, threshold=0.7, async_mode=False)
answer_relevancy_metric = AnswerRelevancyMetric(model=eval_model, threshold=0.7, async_mode=False)
correctness_metric      = GEval(
    name="Correctness",
    criteria="Determina si el actual_output es factualmente correcto comparado con el expected_output.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model=eval_model,
    threshold=0.7,
)


In [38]:
# Test cases con ground truth
test_cases = []
for question, gt in zip(test_questions, ground_truths):
    answer, chunks = ask(question, qdrant, embedding_service)
    test_cases.append(LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[c["content"] for c in chunks],
        expected_output=gt,
    ))

In [39]:
results_gt = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric, answer_relevancy_metric, correctness_metric],
)

✨ You're running DeepEval's latest Faithfulness Metric! (using NVIDIA NIM — nvidia/nemotron-3-super-120b-a12b, 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using NVIDIA NIM — nvidia/nemotron-3-super-120b-a12b,
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using NVIDIA NIM — 
nvidia/nemotron-3-super-120b-a12b, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              ¿Cuál es el precio del plan Enterprise de CloudSync Pro?                             │
│  │     Actual Output:      $49/mes por usuario con compromiso anual.                                            │
│  │     Expected Output:    El plan Enterprise de CloudSync Pro cuesta $49/mes por usuario con compromiso        │
│  │                         anual. Sin compromiso, el plan mensual está disponible a $59/mes por usuario.        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness        │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...     │
│        PASS  │ Answer Relevancy    │ 1.00  │ 0.70      │ The score is 1.00 because the actual output dir...     │
│        FAIL  │ Correctness [GEval] │ 0.60  │ 0.70      │ The actual output correctly states the annual          │
│              │                     │       │           │ commitment price of $49/mes por usuario, which         │
│              │                     │       │           │ matches the expected output. However, it omits the     │
│              │                     │       │           │ additional factual information about the monthly       │
│              │                     │       │           │ plan without commitment being $59/mes por usuario,     │
│              │                     │       │           │ making it incomplete despite not contradicting any     │
│              │                     │       │           │ facts.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              ¿Cuántos días a la semana pueden trabajar remotamente los empleados?                 │
│  │     Actual Output:      Los empleados pueden trabajar de forma remota hasta 3 días por semana.               │
│  │     Expected Output:    Los empleados pueden trabajar de forma remota hasta 3 días por semana, con           │
│  │                         aprobación del jefe directo.                                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness        │ 1.00  │ 0.70      │ 

⚠ WARNING: No hyperparameters logged.
» ]8;id=855932;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 63.06s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 25.0% | Passed: 1 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.